# dots.mocr: unquantized GPU comparison

Runs original pinned Hugging Face weights on a Colab GPU, never on CPU. BF16 when PyTorch reports support (which can include GPU emulation), otherwise FP16. Neither uses 4/8-bit quantization; the run records actual parameter dtypes.

31 matched pages: original multimodal PDF (2), NASA (20), historical diary (5), author-written mathematics (4). The private ZIP contains the exact model-input PNGs from the quantized runs. No baseline predictions or expected answers are sent to the model.

Select **Runtime → Change runtime type → T4 GPU**, then run the code cell and upload the prepared ZIP. Outputs download as a ZIP after all pages have been attempted, with failures retained. SDPA and an unused FlashAttention import shim provide T4 compatibility; runtime precision and parameter types are recorded.

GGUF source lineage is unverified, and hardware/backends differ: this is a configuration comparison, not a controlled causal estimate of quantization alone. The notebook does not start a web server, use SSH, or purchase compute.


In [ ]:
# Exact matched 31-page dots.mocr cloud GPU comparison. No quantization.
import hashlib, pathlib, shutil, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), "Connect a GPU runtime first"
assert torch.cuda.get_device_properties(0).total_memory >= 12 * 1024**3, "Use a >=12 GiB cloud GPU"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "accelerate>=1.10,<2", "pillow>=11,<13"])
from google.colab import files
print("Upload the private matched-colab-inputs-v2.zip supplied with this test.", flush=True)
uploaded = files.upload()
assert len(uploaded) == 1, "Choose exactly the matched input ZIP"
name, payload = next(iter(uploaded.items()))
expected = "236be23e4a8b7ca61989d0c08dd476b6e1eaabaffad2470626f306a0461cf676"
assert hashlib.sha256(payload).hexdigest() == expected, "Matched input ZIP checksum differs"
folder = pathlib.Path("/content/dots_mocr_matched_inputs")
assert not folder.exists(), "Inputs already exist; retain existing outputs before rerunning"
with zipfile.ZipFile(name) as archive:
    for member in archive.infolist():
        path = pathlib.PurePosixPath(member.filename)
        assert not path.is_absolute() and ".." not in path.parts
    archive.extractall(folder)
del uploaded, payload
output = pathlib.Path("/content/dots_mocr_unquantized_gpu_output")
subprocess.check_call([sys.executable, "-u", str(folder / "run_gpu_test.py"), "--inputs", str(folder), "--output", str(output)])
result = shutil.make_archive("/content/dots_mocr_unquantized_gpu_results", "zip", output)
print("All attempted pages saved. Downloading results for local evaluation:", result, flush=True)
files.download(result)
